# Validação de execução do pipeline YouTube



Este notebook localiza uma execução em `/data`, avalia qualidade e consistência das camadas Bronze, Silver e Gold, e salva um relatório em `reports/validacao`.



Para validar dados em outro local, defina a variável de ambiente `DATA_ROOT` antes de abrir o notebook. Para escolher uma execução específica, altere `RUN_ID_OVERRIDE` na célula seguinte.

## 1. Configurar caminhos e dependências



A entrada padrão é `/data`. Quando esse caminho não estiver disponível, como ao abrir o notebook na raiz deste repositório, é usado `data/`. As saídas ficam em `reports/validacao`.

In [1]:
from __future__ import annotations



import json

import os

from datetime import datetime, timezone

from pathlib import Path



import pandas as pd

from IPython.display import display



PROJECT_ROOT = Path.cwd()

DEFAULT_DATA_ROOT = Path('/data')

DATA_ROOT = Path(os.environ.get('DATA_ROOT', DEFAULT_DATA_ROOT))

if not DATA_ROOT.exists():

    DATA_ROOT = PROJECT_ROOT / 'data'



REPORTS_DIR = PROJECT_ROOT / 'reports' / 'validacao'

INVALID_ROWS_DIR = REPORTS_DIR / 'invalidos'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INVALID_ROWS_DIR.mkdir(parents=True, exist_ok=True)



print(f'Diretório de dados: {DATA_ROOT.resolve()}')

print(f'Diretório de relatórios: {REPORTS_DIR.resolve()}')


Diretório de dados: /home/estevam/personal/tcc-engdados/data
Diretório de relatórios: /home/estevam/personal/tcc-engdados/reports/validacao


In [2]:
# Se estiver vazio, seleciona a execução mais recente comum às camadas Bronze e Silver.

RUN_ID_OVERRIDE: str | None = None



def run_ids(layer: str) -> set[str]:

    layer_dir = DATA_ROOT / layer

    if not layer_dir.exists():

        return set()

    return {path.name for path in layer_dir.iterdir() if path.is_dir()}



available_runs = run_ids('bronze') & run_ids('silver')

if not available_runs:

    raise FileNotFoundError(

        f'Não há execuções comuns em {DATA_ROOT / "bronze"} e {DATA_ROOT / "silver"}.'

    )



RUN_ID = RUN_ID_OVERRIDE or max(available_runs)

RUN_PATHS = {

    layer: DATA_ROOT / layer / RUN_ID

    for layer in ('config', 'bronze', 'silver', 'gold')

}



print(f'Execução selecionada: {RUN_ID}')

display(pd.DataFrame({

    'camada': RUN_PATHS.keys(),

    'caminho': [str(path) for path in RUN_PATHS.values()],

    'disponível': [path.exists() for path in RUN_PATHS.values()],

}))


Execução selecionada: 20260929T163948Z


,camada,caminho,disponível
0,config,/home/estevam/personal/tcc-engdados/data/confi...,True
1,bronze,/home/estevam/personal/tcc-engdados/data/bronz...,True
2,silver,/home/estevam/personal/tcc-engdados/data/silve...,True
3,gold,/home/estevam/personal/tcc-engdados/data/gold/...,True


## 2. Localizar e carregar arquivos



O inventário confirma a presença dos artefatos gerados para a execução selecionada. São aceitos JSON, CSV e Parquet.

In [3]:
EXPECTED_FILES = {

    'config': ['channels.json', 'relevance_terms.json'],

    'bronze': ['videos.json', 'comments.json'],

    'silver': [

        'videos_relevantes.parquet', 'videos_descartados.parquet',

        'videos_silver.parquet', 'comments_silver.parquet',

    ],

    'gold': [

        'comments_with_sentiment.parquet', 'serie_temporal_volume.parquet',

        'candidatos_manifestacoes.parquet', 'temas_engajamento.parquet',

    ],

}



inventory = []

for layer, names in EXPECTED_FILES.items():

    for name in names:

        path = RUN_PATHS[layer] / name

        inventory.append({

            'camada': layer,

            'arquivo': name,

            'existe': path.is_file(),

            'tamanho_kb': round(path.stat().st_size / 1024, 1) if path.is_file() else None,

        })


inventory_df = pd.DataFrame(inventory)

display(inventory_df)



missing_files = inventory_df.loc[~inventory_df['existe'], ['camada', 'arquivo']]

if not missing_files.empty:

    print('Arquivos ausentes:')

    display(missing_files)



def load_table(path: Path) -> pd.DataFrame:

    if path.suffix == '.parquet':

        return pd.read_parquet(path)

    if path.suffix == '.csv':

        return pd.read_csv(path)

    if path.suffix == '.json':

        raw = json.loads(path.read_text(encoding='utf-8'))

        if isinstance(raw, list):

            return pd.json_normalize(raw)

        if isinstance(raw, dict):

            for value in raw.values():

                if isinstance(value, list):

                    return pd.json_normalize(value)

            return pd.json_normalize(raw)

    raise ValueError(f'Formato não suportado: {path.suffix}')



TABLE_PATHS = {

    'bronze_videos': RUN_PATHS['bronze'] / 'videos.json',

    'bronze_comments': RUN_PATHS['bronze'] / 'comments.json',

    'silver_videos': RUN_PATHS['silver'] / 'videos_silver.parquet',

    'silver_comments': RUN_PATHS['silver'] / 'comments_silver.parquet',

    'gold_comments_sentiment': RUN_PATHS['gold'] / 'comments_with_sentiment.parquet',

    'gold_temporal': RUN_PATHS['gold'] / 'serie_temporal_volume.parquet',

    'gold_candidates': RUN_PATHS['gold'] / 'candidatos_manifestacoes.parquet',

    'gold_themes': RUN_PATHS['gold'] / 'temas_engajamento.parquet',

}


tables = {

    name: load_table(path)

    for name, path in TABLE_PATHS.items()

    if path.is_file()

}

print(f'{len(tables)} tabelas carregadas.')


,camada,arquivo,existe,tamanho_kb
0,config,channels.json,True,0.1
1,config,relevance_terms.json,True,2.8
2,bronze,videos.json,True,12.7
3,bronze,comments.json,True,431.7
4,silver,videos_relevantes.parquet,True,13.5
5,silver,videos_descartados.parquet,True,9.2
6,silver,videos_silver.parquet,True,13.5
7,silver,comments_silver.parquet,True,86.7
8,gold,comments_with_sentiment.parquet,True,88.0
9,gold,serie_temporal_volume.parquet,True,4.4


8 tabelas carregadas.


## 3. Inspecionar estrutura e tipos dos dados



A inspeção registra volume, nomes de campos, tipos inferidos, uma amostra e estatísticas descritivas de cada tabela carregada.

In [4]:
table_overview = pd.DataFrame([

    {

        'tabela': name,

        'linhas': len(frame),

        'colunas': len(frame.columns),

        'campos': ', '.join(frame.columns),

    }

    for name, frame in tables.items()

])

display(table_overview)



for name, frame in tables.items():

    print(f'\n### {name} ({len(frame):,} linhas)')

    display(pd.DataFrame({

        'campo': frame.columns,

        'tipo': frame.dtypes.astype(str).values,

        'nulos': frame.isna().sum().values,

    }))

    display(frame.head(3))

    display(frame.describe(include='all').T.head(20))


,tabela,linhas,colunas,campos
0,bronze_videos,24,12,"source, category, channel_id, channel_title, v..."
1,bronze_comments,24,3,"video_id, status, comment_count"
2,silver_videos,23,15,"video_id, channel_id, channel_title, source, c..."
3,silver_comments,1121,12,"comment_id, video_id, source, category, parent..."
4,gold_comments_sentiment,1121,14,"comment_id, video_id, source, category, parent..."
5,gold_temporal,13,6,"data, videos_novos, comentarios_novos, likes_t..."
6,gold_candidates,1,12,"candidato, videos_total, likes_videos_total, c..."
7,gold_themes,1,11,"tema, videos_total, comentarios_total, likes_m..."



### bronze_videos (24 linhas)


,campo,tipo,nulos
0,source,str,0
1,category,str,0
2,channel_id,str,0
3,channel_title,str,0
4,video_id,str,0
5,title,str,0
6,published_at,str,0
7,description,str,0
8,view_count,int64,0
9,like_count,int64,0


,source,category,channel_id,channel_title,video_id,title,published_at,description,view_count,like_count,comment_count,privacy_status
0,Ciro Gomes,candidato,UCHFO37KCJlMNUXNK21MV8SQ,Ciro Gomes,9u4xlCKIg4k,ESTÁDIO DA CIDADEPARA RECEBER CIRO.EM PLENO ME...,2026-09-20T19:06:40+00:00,,51296,1975,71,public
1,Ciro Gomes,candidato,UCHFO37KCJlMNUXNK21MV8SQ,Ciro Gomes,bzXwoED6Ujk,CIRO GOMES PARANDO ASSARÉ NESSE EXATO MOMENTO....,2026-09-20T17:45:47+00:00,,19178,1019,41,public
2,Ciro Gomes,candidato,UCHFO37KCJlMNUXNK21MV8SQ,Ciro Gomes,qMMlBdobs0k,MARTINÓPOLE foi outra cidade que saiu às ruas ...,2026-09-20T14:51:12+00:00,Ciro nao teve agenda lá ontem.DEZENAS DE CIDAD...,5638,543,28,public


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
source,24,1,Ciro Gomes,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,24,1,candidato,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_id,24,1,UCHFO37KCJlMNUXNK21MV8SQ,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_title,24,1,Ciro Gomes,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
video_id,24,24,9u4xlCKIg4k,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,24,24,ESTÁDIO DA CIDADEPARA RECEBER CIRO.EM PLENO ME...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
published_at,24,24,2026-09-20T19:06:40+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,24,5,,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
view_count,24.0,NaN,NaN,NaN,19511.041667,13068.130715,2556.0,12344.0,15727.0,22509.75,51296.0
like_count,24.0,NaN,NaN,NaN,846.333333,437.021109,174.0,565.5,772.5,1023.5,2038.0


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
candidato,1,1,Ciro Gomes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
videos_total,1.0,NaN,NaN,NaN,20.0,NaN,20.0,20.0,20.0,20.0,20.0
likes_videos_total,1.0,NaN,NaN,NaN,15891.0,NaN,15891.0,15891.0,15891.0,15891.0,15891.0
comentarios_total,1.0,NaN,NaN,NaN,685.0,NaN,685.0,685.0,685.0,685.0,685.0
likes_comentarios_total,1.0,NaN,NaN,NaN,2707.0,NaN,2707.0,2707.0,2707.0,2707.0,2707.0
sentimento_positivo,1.0,NaN,NaN,NaN,6.0,NaN,6.0,6.0,6.0,6.0,6.0
sentimento_negativo,1.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0
sentimento_neutro,1.0,NaN,NaN,NaN,679.0,NaN,679.0,679.0,679.0,679.0,679.0
score_sentimento_medio,1.0,NaN,NaN,NaN,0.003102,NaN,0.003102,0.003102,0.003102,0.003102,0.003102
sentimento_positivo_pct,1.0,NaN,NaN,NaN,0.88,NaN,0.88,0.88,0.88,0.88,0.88



### gold_themes (1 linhas)


,campo,tipo,nulos
0,tema,str,0
1,videos_total,float64,0
2,comentarios_total,float64,0
3,likes_media,float64,0
4,sentimento_positivo,float64,0
5,sentimento_negativo,float64,0
6,sentimento_neutro,float64,0
7,score_sentimento_medio,float64,0
8,sentimento_positivo_pct,float64,0
9,sentimento_negativo_pct,float64,0


,tema,videos_total,comentarios_total,likes_media,sentimento_positivo,sentimento_negativo,sentimento_neutro,score_sentimento_medio,sentimento_positivo_pct,sentimento_negativo_pct,sentimento_neutro_pct
0,Ciro Gomes,23.0,1121.0,3.47636,11.0,1.0,1109.0,0.001873,0.98,0.09,98.93


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
tema,1,1,Ciro Gomes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
videos_total,1.0,NaN,NaN,NaN,23.0,NaN,23.0,23.0,23.0,23.0,23.0
comentarios_total,1.0,NaN,NaN,NaN,1121.0,NaN,1121.0,1121.0,1121.0,1121.0,1121.0
likes_media,1.0,NaN,NaN,NaN,3.47636,NaN,3.47636,3.47636,3.47636,3.47636,3.47636
sentimento_positivo,1.0,NaN,NaN,NaN,11.0,NaN,11.0,11.0,11.0,11.0,11.0
sentimento_negativo,1.0,NaN,NaN,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0
sentimento_neutro,1.0,NaN,NaN,NaN,1109.0,NaN,1109.0,1109.0,1109.0,1109.0,1109.0
score_sentimento_medio,1.0,NaN,NaN,NaN,0.001873,NaN,0.001873,0.001873,0.001873,0.001873,0.001873
sentimento_positivo_pct,1.0,NaN,NaN,NaN,0.98,NaN,0.98,0.98,0.98,0.98,0.98
sentimento_negativo_pct,1.0,NaN,NaN,NaN,0.09,NaN,0.09,0.09,0.09,0.09,0.09


## 4. Validar campos obrigatórios e valores nulos



As regras abaixo distinguem uma coluna ausente de valores nulos. Identificadores, campos de contexto e métricas essenciais são considerados críticos.

In [5]:
checks: list[dict[str, object]] = []

invalid_rows: dict[str, pd.DataFrame] = {}



def register_check(

    rule: str,

    invalid_mask: pd.Series | None,

    frame: pd.DataFrame | None = None,

    severity: str = 'ERRO',

    details: str = '',

) -> None:

    invalid_count = int(invalid_mask.sum()) if invalid_mask is not None else 1

    total = len(frame) if frame is not None else 1

    status = 'APROVADO' if invalid_count == 0 else severity

    checks.append({

        'regra': rule,

        'status': status,

        'registros_invalidos': invalid_count,

        'percentual_afetado': round((invalid_count / total) * 100, 2),

        'detalhes': details,

    })

    if frame is not None and invalid_mask is not None and invalid_count:

        invalid_rows[rule] = frame.loc[invalid_mask].copy()



REQUIRED_COLUMNS = {

    'bronze_videos': ['video_id', 'channel_id', 'source', 'title', 'published_at'],

    'bronze_comments': ['video_id', 'status', 'comment_count'],

    'silver_videos': ['video_id', 'source', 'published_at', 'processed_at'],

    'silver_comments': ['comment_id', 'video_id', 'text_clean', 'published_at'],

    'gold_comments_sentiment': ['comment_id', 'video_id', 'sentimento', 'score_sentimento'],

    'gold_temporal': ['data', 'videos_novos', 'comentarios_novos'],

    'gold_candidates': ['candidato', 'videos_total', 'comentarios_total'],

    'gold_themes': ['tema', 'videos_total', 'comentarios_total'],

}



for table_name, columns in REQUIRED_COLUMNS.items():

    frame = tables.get(table_name)

    if frame is None:

        register_check(f'{table_name}: arquivo disponível', None, severity='ERRO', details='Tabela não carregada.')

        continue

    missing_columns = sorted(set(columns) - set(frame.columns))

    register_check(

        f'{table_name}: campos obrigatórios',

        pd.Series([bool(missing_columns)]),

        severity='ERRO',

        details=', '.join(missing_columns) if missing_columns else 'Todos presentes.',

    )

    for column in set(columns) & set(frame.columns):

        null_mask = frame[column].isna() | frame[column].astype('string').str.strip().eq('')

        register_check(f'{table_name}.{column}: valores obrigatórios', null_mask, frame)



display(pd.DataFrame(checks))


,regra,status,registros_invalidos,percentual_afetado,detalhes
0,bronze_videos: campos obrigatórios,APROVADO,0,0.0,Todos presentes.
1,bronze_videos.source: valores obrigatórios,APROVADO,0,0.0,
2,bronze_videos.video_id: valores obrigatórios,APROVADO,0,0.0,
3,bronze_videos.published_at: valores obrigatórios,APROVADO,0,0.0,
4,bronze_videos.title: valores obrigatórios,APROVADO,0,0.0,
5,bronze_videos.channel_id: valores obrigatórios,APROVADO,0,0.0,
6,bronze_comments: campos obrigatórios,APROVADO,0,0.0,Todos presentes.
7,bronze_comments.status: valores obrigatórios,APROVADO,0,0.0,
8,bronze_comments.comment_count: valores obrigat...,APROVADO,0,0.0,
9,bronze_comments.video_id: valores obrigatórios,APROVADO,0,0.0,


## 5. Validar tipos, faixas e regras de domínio



As regras verificam identificadores, métricas numéricas, valores não negativos e o domínio do sentimento calculado na camada Gold.

In [6]:
ID_RULES = {

    'bronze_videos': {'video_id': 8, 'channel_id': 8},

    'bronze_comments': {'video_id': 8},

    'silver_videos': {'video_id': 8},

    'silver_comments': {'comment_id': 8, 'video_id': 8},

    'gold_comments_sentiment': {'comment_id': 8, 'video_id': 8},

}

NON_NEGATIVE_COLUMNS = {

    'bronze_videos': ['view_count', 'like_count', 'comment_count'],

    'bronze_comments': ['comment_count'],

    'silver_videos': ['view_count', 'like_count', 'comment_count'],

    'silver_comments': ['like_count'],

    'gold_temporal': ['videos_novos', 'comentarios_novos', 'likes_total', 'engajamento_medio'],

    'gold_candidates': ['videos_total', 'comentarios_total', 'likes_videos_total'],

    'gold_themes': ['videos_total', 'comentarios_total', 'likes_media'],

}



for table_name, rules in ID_RULES.items():

    frame = tables.get(table_name)

    if frame is None:

        continue

    for column, min_length in rules.items():

        if column in frame:

            invalid_mask = frame[column].astype('string').str.strip().str.len().lt(min_length)

            register_check(f'{table_name}.{column}: identificador válido', invalid_mask, frame)



for table_name, columns in NON_NEGATIVE_COLUMNS.items():

    frame = tables.get(table_name)

    if frame is None:

        continue

    for column in set(columns) & set(frame.columns):

        values = pd.to_numeric(frame[column], errors='coerce')

        invalid_mask = values.isna() | values.lt(0)

        register_check(f'{table_name}.{column}: número não negativo', invalid_mask, frame)


sentiment = tables.get('gold_comments_sentiment')

if sentiment is not None:

    allowed_sentiments = {'POSITIVO', 'NEGATIVO', 'NEUTRO'}

    register_check(

        'gold_comments_sentiment.sentimento: domínio permitido',

        ~sentiment['sentimento'].isin(allowed_sentiments),

        sentiment,

        details=', '.join(sorted(allowed_sentiments)),

    )

    scores = pd.to_numeric(sentiment['score_sentimento'], errors='coerce')

    register_check(

        'gold_comments_sentiment.score_sentimento: faixa [-1, 1]',

        scores.isna() | ~scores.between(-1, 1),

        sentiment,

    )


display(pd.DataFrame(checks).tail(20))


,regra,status,registros_invalidos,percentual_afetado,detalhes
45,bronze_videos.like_count: número não negativo,APROVADO,0,0.0,
46,bronze_videos.view_count: número não negativo,APROVADO,0,0.0,
47,bronze_videos.comment_count: número não negativo,APROVADO,0,0.0,
48,bronze_comments.comment_count: número não nega...,APROVADO,0,0.0,
49,silver_videos.like_count: número não negativo,APROVADO,0,0.0,
50,silver_videos.view_count: número não negativo,APROVADO,0,0.0,
51,silver_videos.comment_count: número não negativo,APROVADO,0,0.0,
52,silver_comments.like_count: número não negativo,APROVADO,0,0.0,
53,gold_temporal.videos_novos: número não negativo,APROVADO,0,0.0,
54,gold_temporal.engajamento_medio: número não ne...,APROVADO,0,0.0,


## 6. Detectar registros duplicados e inconsistências



A verificação usa as chaves de negócio de vídeos e comentários, relaciona as camadas e testa a coerência dos percentuais agregados.

In [7]:
PRIMARY_KEYS = {

    'bronze_videos': ['video_id'],

    'bronze_comments': ['video_id'],

    'silver_videos': ['video_id'],

    'silver_comments': ['comment_id'],

    'gold_comments_sentiment': ['comment_id'],

    'gold_temporal': ['data'],

    'gold_candidates': ['candidato'],

    'gold_themes': ['tema'],

}



for table_name, key in PRIMARY_KEYS.items():

    frame = tables.get(table_name)

    if frame is not None and set(key).issubset(frame.columns):

        register_check(

            f'{table_name}: chave única ({", ".join(key)})',

            frame.duplicated(key, keep=False),

            frame,

        )



if {'bronze_videos', 'silver_videos'} <= tables.keys():

    bronze_ids = set(tables['bronze_videos']['video_id'])

    silver_videos = tables['silver_videos']

    register_check(

        'silver_videos: video_id existe no Bronze',

        ~silver_videos['video_id'].isin(bronze_ids),

        silver_videos,

    )



if {'silver_videos', 'silver_comments'} <= tables.keys():

    silver_video_ids = set(tables['silver_videos']['video_id'])

    silver_comments = tables['silver_comments']

    register_check(

        'silver_comments: video_id existe em silver_videos',

        ~silver_comments['video_id'].isin(silver_video_ids),

        silver_comments,

    )



if {'silver_comments', 'gold_comments_sentiment'} <= tables.keys():

    silver_comment_ids = set(tables['silver_comments']['comment_id'])

    gold_comments = tables['gold_comments_sentiment']

    register_check(

        'gold_comments_sentiment: comment_id existe em silver_comments',

        ~gold_comments['comment_id'].isin(silver_comment_ids),

        gold_comments,

    )

    register_check(

        'gold_comments_sentiment: mesma quantidade de comentários Silver',

        pd.Series([len(gold_comments) != len(silver_comment_ids)]),

        severity='ERRO',

        details=f'Gold={len(gold_comments)}, Silver={len(silver_comment_ids)}',

    )



for table_name in ('gold_candidates', 'gold_themes'):

    frame = tables.get(table_name)

    pct_columns = [

        'sentimento_positivo_pct', 'sentimento_negativo_pct', 'sentimento_neutro_pct',

    ]

    if frame is not None and set(pct_columns).issubset(frame.columns):

        total_pct = frame[pct_columns].sum(axis=1)

        register_check(

            f'{table_name}: percentuais de sentimento somam 100',

            ~total_pct.between(99.99, 100.01),

            frame,

        )


display(pd.DataFrame(checks).tail(20))


,regra,status,registros_invalidos,percentual_afetado,detalhes
59,gold_candidates.comentarios_total: número não ...,APROVADO,0,0.0,
60,gold_themes.videos_total: número não negativo,APROVADO,0,0.0,
61,gold_themes.likes_media: número não negativo,APROVADO,0,0.0,
62,gold_themes.comentarios_total: número não nega...,APROVADO,0,0.0,
63,gold_comments_sentiment.sentimento: domínio pe...,APROVADO,0,0.0,"NEGATIVO, NEUTRO, POSITIVO"
64,gold_comments_sentiment.score_sentimento: faix...,APROVADO,0,0.0,
65,bronze_videos: chave única (video_id),APROVADO,0,0.0,
66,bronze_comments: chave única (video_id),APROVADO,0,0.0,
67,silver_videos: chave única (video_id),APROVADO,0,0.0,
68,silver_comments: chave única (comment_id),APROVADO,0,0.0,


## 7. Validar datas e sequência temporal



Datas são convertidas de forma tolerante para identificar valores inválidos, futuros e relações temporais incoerentes.

In [8]:
DATE_COLUMNS = {

    'bronze_videos': ['published_at'],

    'silver_videos': ['published_at', 'processed_at'],

    'silver_comments': ['published_at', 'updated_at', 'processed_at'],

    'gold_comments_sentiment': ['published_at', 'updated_at', 'processed_at'],

    'gold_temporal': ['data'],

}

now_utc = pd.Timestamp.now(tz='UTC')



for table_name, columns in DATE_COLUMNS.items():

    frame = tables.get(table_name)

    if frame is None:

        continue

    for column in set(columns) & set(frame.columns):

        values = pd.to_datetime(frame[column], errors='coerce', utc=True)

        register_check(f'{table_name}.{column}: data válida', values.isna(), frame)

        register_check(

            f'{table_name}.{column}: não está no futuro',

            values.gt(now_utc),

            frame,

            severity='AVISO',

            details=f'Referência: {now_utc.isoformat()}',

        )



for table_name in ('silver_comments', 'gold_comments_sentiment'):

    frame = tables.get(table_name)

    if frame is not None and {'published_at', 'updated_at'} <= set(frame.columns):

        published = pd.to_datetime(frame['published_at'], errors='coerce', utc=True)

        updated = pd.to_datetime(frame['updated_at'], errors='coerce', utc=True)

        register_check(

            f'{table_name}: updated_at não antecede published_at',

            updated.lt(published),

            frame,

        )


temporal = tables.get('gold_temporal')

if temporal is not None and 'data' in temporal:

    temporal_dates = pd.to_datetime(temporal['data'], errors='coerce')

    register_check(

        'gold_temporal: datas em ordem crescente',

        pd.Series([not temporal_dates.is_monotonic_increasing]),

        severity='AVISO',

    )


display(pd.DataFrame(checks).tail(20))


,regra,status,registros_invalidos,percentual_afetado,detalhes
82,silver_videos.processed_at: não está no futuro,APROVADO,0,0.0,Referência: 2026-09-29T16:45:34.528858+00:00
83,silver_videos.published_at: data válida,APROVADO,0,0.0,
84,silver_videos.published_at: não está no futuro,APROVADO,0,0.0,Referência: 2026-09-29T16:45:34.528858+00:00
85,silver_comments.processed_at: data válida,APROVADO,0,0.0,
86,silver_comments.processed_at: não está no futuro,APROVADO,0,0.0,Referência: 2026-09-29T16:45:34.528858+00:00
87,silver_comments.published_at: data válida,APROVADO,0,0.0,
88,silver_comments.published_at: não está no futuro,APROVADO,0,0.0,Referência: 2026-09-29T16:45:34.528858+00:00
89,silver_comments.updated_at: data válida,APROVADO,0,0.0,
90,silver_comments.updated_at: não está no futuro,APROVADO,0,0.0,Referência: 2026-09-29T16:45:34.528858+00:00
91,gold_comments_sentiment.processed_at: data válida,APROVADO,0,0.0,


## 8. Gerar relatório de qualidade dos dados



O relatório consolida regra, status, número e percentual de registros afetados, detalhes e uma referência aos exemplos capturados.

In [9]:
validation_report = pd.DataFrame(checks)

validation_report['exemplos'] = validation_report['regra'].map(

    lambda rule: f'{rule}.csv' if rule in invalid_rows else ''

)

validation_report = validation_report.sort_values(['status', 'regra']).reset_index(drop=True)



summary = (

    validation_report.groupby('status', dropna=False)

    .agg(regras=('regra', 'size'), registros_invalidos=('registros_invalidos', 'sum'))

    .reset_index()

)

display(summary)

display(validation_report)


,status,regras,registros_invalidos
0,APROVADO,102,0


,regra,status,registros_invalidos,percentual_afetado,detalhes,exemplos
0,bronze_comments.comment_count: número não nega...,APROVADO,0,0.0,,
1,bronze_comments.comment_count: valores obrigat...,APROVADO,0,0.0,,
2,bronze_comments.status: valores obrigatórios,APROVADO,0,0.0,,
3,bronze_comments.video_id: identificador válido,APROVADO,0,0.0,,
4,bronze_comments.video_id: valores obrigatórios,APROVADO,0,0.0,,
...,...,...,...,...,...,...
97,silver_videos.video_id: valores obrigatórios,APROVADO,0,0.0,,
98,silver_videos.view_count: número não negativo,APROVADO,0,0.0,,
99,silver_videos: campos obrigatórios,APROVADO,0,0.0,Todos presentes.,
100,silver_videos: chave única (video_id),APROVADO,0,0.0,,


## 9. Salvar resultados da validação



O relatório é salvo em CSV e JSON. Cada regra com evidências exporta suas linhas inválidas em arquivo separado; o status final falha somente quando há erros críticos.

In [10]:
run_report_dir = REPORTS_DIR / RUN_ID

run_invalid_dir = run_report_dir / 'invalidos'

run_invalid_dir.mkdir(parents=True, exist_ok=True)



report_csv = run_report_dir / 'relatorio_validacao.csv'

report_json = run_report_dir / 'relatorio_validacao.json'

validation_report.to_csv(report_csv, index=False)

validation_report.to_json(report_json, orient='records', force_ascii=False, indent=2)



invalid_exports = []

for rule, frame in invalid_rows.items():

    safe_name = ''.join(char if char.isalnum() else '_' for char in rule).strip('_')

    output_file = run_invalid_dir / f'{safe_name}.csv'

    frame.to_csv(output_file, index=False)

    invalid_exports.append(output_file)



critical_errors = validation_report['status'].eq('ERRO').sum()

warnings = validation_report['status'].eq('AVISO').sum()

final_status = 'FALHOU' if critical_errors else 'APROVADO'



print(f'Status final: {final_status}')

print(f'Erros críticos: {critical_errors} | Avisos: {warnings}')

print(f'Relatório CSV: {report_csv}')

print(f'Relatório JSON: {report_json}')

print(f'Arquivos com registros inválidos: {len(invalid_exports)}')


Status final: APROVADO
Erros críticos: 0 | Avisos: 0
Relatório CSV: /home/estevam/personal/tcc-engdados/reports/validacao/20260929T163948Z/relatorio_validacao.csv
Relatório JSON: /home/estevam/personal/tcc-engdados/reports/validacao/20260929T163948Z/relatorio_validacao.json
Arquivos com registros inválidos: 0
